# PPO CNN — clean 6×6 validation run

This notebook trains the scalable CNN policy on **6×6 beams** for exactly **5,000 vector steps** with 64 asynchronous environments (up to 320,000 environment transitions). It keeps training, diagnostics, checkpoints, and rendering separate so reruns are predictable.

The default device mode is `cuda`: if CUDA is unhealthy, the notebook stops immediately instead of silently running a long CPU job. Change it to `auto` or `cpu` only when that fallback is intentional.

In [ ]:
from __future__ import annotations

import csv
import math
import random
import time
import warnings
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.multiprocessing as mp
from gymnasium.vector import AsyncVectorEnv
from IPython.display import display

from env import mbb_cnn
from models import ppo_cnn

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titleweight': 'bold',
})

## Configuration

`TOTAL_VECTOR_STEPS` counts calls to the vector environment, matching the existing notebook. Each call advances all 64 environments once.

In [ ]:
GRID_WIDTH = 6
GRID_HEIGHT = 6
TARGET_DENSITY = 0.4
TOTAL_VECTOR_STEPS = 5_000
NUM_ENVS = 64
ROLLOUT_STEPS = 128
CHECKPOINT_EVERY = 1_000
REPORT_EVERY = 250
SEED = 0

USE_RANDOM_BEAMS = True
USE_TWO_PATH_ARCH = True
DEVICE_MODE = 'cuda'  # 'cuda' fails fast; 'auto' permits CPU fallback; 'cpu' forces CPU
EVAL_BEAM_TYPES = tuple(range(1, 10))
RANDOM_EVAL_TRIALS = 5

MAX_EPISODE_STEPS = 3 * GRID_WIDTH * GRID_HEIGHT
TRANSITION_BUDGET = TOTAL_VECTOR_STEPS * NUM_ENVS
assert GRID_WIDTH > 0 and GRID_HEIGHT > 0
assert 0.001 < TARGET_DENSITY < 1.0

print(f'Grid: {GRID_WIDTH}×{GRID_HEIGHT}')
print(f'Training: {TOTAL_VECTOR_STEPS:,} vector steps × {NUM_ENVS} envs = {TRANSITION_BUDGET:,} transitions')
print(f'Episode horizon: {MAX_EPISODE_STEPS} actions (construction + refinement)')

## Device check

If this cell reports `CUDA_ERROR_UNKNOWN`, stop the notebook kernel and reload the NVIDIA UVM module in a terminal with `sudo modprobe -r nvidia_uvm && sudo modprobe nvidia_uvm`. Then start a fresh kernel and rerun this cell.

In [ ]:
def resolve_device(mode: str) -> torch.device:
    mode = mode.lower().strip()
    if mode not in {'cuda', 'auto', 'cpu'}:
        raise ValueError("DEVICE_MODE must be 'cuda', 'auto', or 'cpu'")
    if mode == 'cpu':
        return torch.device('cpu')

    cuda_error = None
    try:
        if not torch.cuda.is_available():
            raise RuntimeError('torch.cuda.is_available() returned False')
        candidate = torch.device('cuda:0')
        probe = torch.ones(1, device=candidate)
        del probe
        torch.cuda.empty_cache()
        return candidate
    except Exception as error:
        cuda_error = error

    message = (
        'CUDA was requested but could not initialize. Stop the notebook kernel, '
        'reload nvidia_uvm, then start a fresh kernel. Root cause: '
        f'{type(cuda_error).__name__}: {cuda_error}'
    )
    if mode == 'cuda':
        raise RuntimeError(message) from cuda_error
    warnings.warn(message + ' — falling back to CPU.', RuntimeWarning)
    return torch.device('cpu')


device = resolve_device(DEVICE_MODE)
if device.type == 'cuda':
    print(f'Device: {torch.cuda.get_device_name(device)}')
    print(f'PyTorch {torch.__version__} | CUDA build {torch.version.cuda}')
else:
    print('Device: CPU (intentional fallback)')

## Environment and policy

In [ ]:
def make_training_env():
    return mbb_cnn.BeamOptimizationEnv(
        width=GRID_WIDTH,
        height=GRID_HEIGHT,
        density=TARGET_DENSITY,
        reward_weights=None,
        use_random_beam=USE_RANDOM_BEAMS,
        beam_type=1,
    )


mp.set_start_method('spawn', force=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if device.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)

probe_env = make_training_env()
state_shape = probe_env.observation_space.shape
action_dim = probe_env.action_space.n
probe_env.close()

ppo_agent = ppo_cnn.PPO(
    state_channels=state_shape[0],
    width=state_shape[1],
    height=state_shape[2],
    action_dim=action_dim,
    lr_actor=3e-4,
    lr_critic=1e-3,
    gamma=0.95,
    K_epochs=4,
    eps_clip=0.2,
    has_continuous_action_space=False,
    gae_lambda=0.95,
    num_envs=NUM_ENVS,
    device=device,
    use_two_path=USE_TWO_PATH_ARCH,
    rollout_steps=ROLLOUT_STEPS,
    minibatch_size=1024,
    target_density=TARGET_DENSITY,
)

parameter_count = sum(parameter.numel() for parameter in ppo_agent.policy.parameters())
print(f'Observation: {state_shape} | actions: {action_dim}')
print(f'Policy parameters: {parameter_count:,}')

## Train

The loop stops at exactly 5,000 vector steps, preserves PPO rollouts across episode boundaries, bootstraps non-terminal rollouts, trains the final partial rollout, and always closes worker processes.

In [ ]:
def train(agent):
    run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
    run_name = f'BeamOptimizationEnv_{GRID_WIDTH}x{GRID_HEIGHT}_{run_id}'
    log_dir = Path('PPO_logs') / run_name
    checkpoint_dir = Path('PPO_preTrained') / run_name
    log_dir.mkdir(parents=True, exist_ok=False)
    checkpoint_dir.mkdir(parents=True, exist_ok=False)
    csv_path = log_dir / 'training.csv'
    checkpoint_path = checkpoint_dir / 'policy.pth'

    history = {'episode': [], 'vector_step': [], 'mean_return': []}
    update_history = []
    vector_step = 0
    episode = 0
    started = time.perf_counter()
    vector_env = None
    states = None

    try:
        vector_env = AsyncVectorEnv([make_training_env for _ in range(NUM_ENVS)])
        with csv_path.open('w', newline='') as log_file:
            writer = csv.DictWriter(log_file, fieldnames=history.keys())
            writer.writeheader()

            while vector_step < TOTAL_VECTOR_STEPS:
                reset_seed = SEED if episode == 0 else None
                states_np, _ = vector_env.reset(seed=reset_seed)
                states = torch.as_tensor(states_np, dtype=torch.float32, device=device)
                episode_returns = np.zeros(NUM_ENVS, dtype=np.float64)
                dones = np.zeros(NUM_ENVS, dtype=bool)
                steps_this_episode = min(
                    MAX_EPISODE_STEPS, TOTAL_VECTOR_STEPS - vector_step
                )

                for _ in range(steps_this_episode):
                    actions, logprobs, state_values = agent.select_action(states)
                    next_states_np, rewards, terminated, truncated, _ = vector_env.step(
                        actions.cpu().numpy()
                    )
                    dones = np.logical_or(terminated, truncated)
                    next_states = torch.as_tensor(
                        next_states_np, dtype=torch.float32, device=device
                    )
                    agent.buffer.store(
                        states=states,
                        actions=actions,
                        logprobs=logprobs,
                        rewards=torch.as_tensor(rewards, dtype=torch.float32, device=device),
                        state_values=state_values,
                        is_terminals=torch.as_tensor(dones, dtype=torch.bool, device=device),
                    )

                    states = next_states
                    episode_returns += rewards
                    vector_step += 1

                    if agent.buffer.is_full:
                        metrics = agent.update(agent.estimate_values(states))
                        update_history.append({'vector_step': vector_step, **metrics})

                    if vector_step % CHECKPOINT_EVERY == 0:
                        agent.save(checkpoint_path)

                    if vector_step % REPORT_EVERY == 0:
                        elapsed = time.perf_counter() - started
                        print(
                            f'{vector_step:5d}/{TOTAL_VECTOR_STEPS:,} steps | '
                            f'current mean return {episode_returns.mean():8.3f} | '
                            f'{elapsed:7.1f}s'
                        )

                    if dones.all():
                        break

                if dones.all():
                    row = {
                        'episode': episode,
                        'vector_step': vector_step,
                        'mean_return': float(episode_returns.mean()),
                    }
                    for key in history:
                        history[key].append(row[key])
                    writer.writerow(row)
                    log_file.flush()
                episode += 1

        if agent.buffer.ptr > 0:
            metrics = agent.update(agent.estimate_values(states))
            update_history.append({'vector_step': vector_step, **metrics})
        agent.save(checkpoint_path)
    finally:
        if vector_env is not None:
            vector_env.close()

    elapsed = time.perf_counter() - started
    print(f'Finished {vector_step:,} vector steps in {elapsed / 60:.1f} min')
    print(f'Checkpoint: {checkpoint_path}')
    print(f'Log: {csv_path}')
    return history, update_history, checkpoint_path


training_history, update_history, checkpoint_path = train(ppo_agent)

## Training diagnostics

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].plot(
    training_history['vector_step'],
    training_history['mean_return'],
    color='#5b2a86',
    linewidth=1.8,
)
axes[0].set(title='Mean vector-episode return', xlabel='Vector step', ylabel='Return')
axes[0].grid(alpha=0.25)

if update_history:
    update_frame = pd.DataFrame(update_history)
    axes[1].plot(update_frame['vector_step'], update_frame['actor_loss'], label='Actor')
    axes[1].plot(update_frame['vector_step'], update_frame['critic_loss'], label='Critic')
    axes[1].legend(frameon=False)
axes[1].set(title='PPO losses', xlabel='Vector step', ylabel='Loss')
axes[1].grid(alpha=0.25)
figure.suptitle(f'PPO training — {GRID_WIDTH}×{GRID_HEIGHT}, {NUM_ENVS} environments', fontsize=14)
figure.tight_layout()
plt.show()

## Deterministic evaluation and rendering

Purple is void and yellow is solid. Cyan squares mark supports; red arrows mark loads. Every panel uses the same fixed 0–1 density scale.

In [ ]:
def evaluate_policy(agent, *, beam_type=None, random_beam=False, seed=None):
    env = mbb_cnn.BeamOptimizationEnv(
        width=GRID_WIDTH,
        height=GRID_HEIGHT,
        density=TARGET_DENSITY,
        reward_weights=None,
        beam_type=1 if beam_type is None else beam_type,
        use_random_beam=random_beam,
    )
    try:
        observation, info = env.reset(seed=seed)
        total_reward = 0.0
        steps = 0
        agent.policy_old.eval()
        for steps in range(1, MAX_EPISODE_STEPS + 1):
            state = torch.as_tensor(
                observation, dtype=torch.float32, device=device
            ).unsqueeze(0)
            action, _, _ = agent.select_action(state, deterministic=True)
            observation, reward, terminated, truncated, info = env.step(
                int(action.item())
            )
            total_reward += float(reward)
            if terminated or truncated:
                break

        return {
            'label': 'Random' if random_beam else f'Beam {beam_type}',
            'density': env.density_field(),
            'normals': env.normals.cpu().numpy().copy(),
            'forces': env.forces.cpu().numpy().copy(),
            'reward': total_reward,
            'compliance': float(env.current_compliance),
            'compliance_ratio': float(env.current_compliance / env.base_compliance_value),
            'volume': float(env.current_constraint),
            'objective': float(env.current_objective),
            'steps': steps,
        }
    finally:
        env.close()


def render_gallery(results, title, columns=3):
    rows = math.ceil(len(results) / columns)
    figure, axes = plt.subplots(
        rows, columns, figsize=(4.5 * columns, 4.1 * rows), squeeze=False
    )
    image = None
    for axis, result in zip(axes.flat, results):
        density = result['density']
        height, width = density.shape
        image = axis.imshow(
            density,
            cmap='viridis',
            interpolation='nearest',
            vmin=0.0,
            vmax=1.0,
            extent=(0, width, height, 0),
        )

        support_x, support_y = np.where(result['normals'].any(axis=2))
        axis.scatter(
            support_x, support_y, marker='s', s=55, facecolors='none',
            edgecolors='#00ffff', linewidths=1.8, label='Support'
        )
        load_x, load_y = np.where(np.linalg.norm(result['forces'], axis=2) > 0)
        for x_coord, y_coord in zip(load_x, load_y):
            force_x, force_y = result['forces'][x_coord, y_coord]
            axis.annotate(
                '',
                xy=(x_coord + 0.65 * force_x, y_coord - 0.65 * force_y),
                xytext=(x_coord, y_coord),
                arrowprops={'arrowstyle': '-|>', 'color': '#ff334d', 'lw': 2.2},
            )

        axis.set_xticks(np.arange(width + 1))
        axis.set_yticks(np.arange(height + 1))
        axis.grid(color='white', alpha=0.32, linewidth=0.7)
        axis.set_aspect('equal')
        axis.set_title(
            f"{result['label']} | C/C₀={result['compliance_ratio']:.3g}\n"
            f"V={result['volume']:.3f} | R={result['reward']:.2f}"
        )
        axis.set_xlabel('X')
        axis.set_ylabel('Y')

    for axis in axes.flat[len(results):]:
        axis.set_visible(False)
    figure.suptitle(title, fontsize=16, y=1.01)
    figure.colorbar(image, ax=list(axes.flat), shrink=0.78, pad=0.02, label='Density')
    plt.show()


fixed_results = [
    evaluate_policy(ppo_agent, beam_type=beam_type)
    for beam_type in EVAL_BEAM_TYPES
]
random_results = [
    evaluate_policy(ppo_agent, random_beam=True, seed=10_000 + trial)
    for trial in range(RANDOM_EVAL_TRIALS)
]
for trial, result in enumerate(random_results, start=1):
    result['label'] = f'Random {trial}'

summary = pd.DataFrame(fixed_results + random_results).drop(
    columns=['density', 'normals', 'forces']
)
display(summary.style.format({
    'reward': '{:.3f}',
    'compliance': '{:.3f}',
    'compliance_ratio': '{:.4f}',
    'volume': '{:.4f}',
    'objective': '{:.4f}',
}).background_gradient(subset=['compliance_ratio'], cmap='viridis_r'))

render_gallery(fixed_results, 'Fixed-beam generalization (1–9)', columns=3)
render_gallery(random_results, 'Random-beam generalization', columns=3)